In [ ]:
# Synchronized regression pipeline with XGBoost (if installed) and best-model selection
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree
import joblib
import folium
import warnings
warnings.filterwarnings("ignore")

# Optional XGBoost
try:
    from xgboost import XGBRegressor
    xgb_available = True
except Exception:
    xgb_available = False

# ---------- CONFIG ----------
SESSIONS_CSV = r"C:\Users\Yash\OneDrive\Desktop\Yash\Evision\Evision\Files\Code Generated CSV\sessions_master.csv"
STATIONS_CSV = r"C:\Users\Yash\OneDrive\Desktop\Yash\Evision\Evision\Files\Code Generated CSV\station_aggregates.csv"

OUTPUT_MODEL = "Model.pkl"
OUTPUT_SUGGESTIONS = "Predicted_Locations.csv"
OUTPUT_MAP = "Predicted_Locations_Map"".html"
OUTPUT_MODEL_COMP = "regression_model_comparison_results.csv"

MIN_DISTANCE_FOR_FAR_KM = 2.0
MAX_NEAR_DISTANCE_KM = 0.5
TOP_N_PER_BUCKET = 100
NEARBY_RADIUS_KM = 2.0
RANDOM_STATE = 42
N_FOLDS = 5

# ---------- HELPERS ----------
def find_latlon_columns(df):
    lat_candidates = ["latitude","lat","Latitude","LAT"]
    lon_candidates = ["longitude","lon","Longitude","LON","lng","Lng"]
    for lat in lat_candidates:
        for lon in lon_candidates:
            if lat in df.columns and lon in df.columns:
                return lat, lon
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric) >= 2:
        return numeric[0], numeric[1]
    raise ValueError("No lat/lon columns found.")

def haversine_balltree_min_dist_km(points_deg, existing_deg):
    if existing_deg.shape[0] == 0:
        return np.full(points_deg.shape[0], np.inf)
    earth_r = 6371.0
    tree = BallTree(np.radians(existing_deg), metric='haversine')
    dist_rad, _ = tree.query(np.radians(points_deg), k=1)
    return (dist_rad.flatten() * earth_r)

def haversine_counts_within(points_deg, other_deg, radius_km):
    if other_deg.shape[0] == 0:
        return np.zeros(points_deg.shape[0], dtype=int)
    earth_r = 6371.0
    tree = BallTree(np.radians(other_deg), metric='haversine')
    rad = radius_km / earth_r
    ind = tree.query_radius(np.radians(points_deg), r=rad)
    return np.array([len(a) for a in ind])

# ---------- 1) Load data ----------
if not os.path.exists(SESSIONS_CSV) or not os.path.exists(STATIONS_CSV):
    raise FileNotFoundError("Make sure both CSV paths are correct.")

sessions = pd.read_csv(SESSIONS_CSV)
stations = pd.read_csv(STATIONS_CSV)

s_lat, s_lon = find_latlon_columns(sessions)
e_lat, e_lon = find_latlon_columns(stations)
print(f"Session coords: {s_lat},{s_lon} | Station coords: {e_lat},{e_lon}")
print(f"Rows -> sessions: {len(sessions)}, stations: {len(stations)}")

# ---------- 2) Choose possible targets (derived from uploaded station_aggregates) ----------
# Use the most relevant demand columns if present
possible_targets = [
    "avg_sessions_per_month",
    "total_sessions",
    "total_energy_kWh",
    "avg_energy_per_session_kWh",
    "sessions_count",        # fallback checks
    "sessions", "session_count"
]
target_col = None
for c in possible_targets:
    if c in stations.columns:
        target_col = c
        break
if target_col is None:
    # fallback: pick first numeric column that looks like demand
    for c in stations.select_dtypes(include=[np.number]).columns:
        if c not in [e_lat, e_lon]:
            target_col = c
            break
if target_col is None:
    raise ValueError("No target column found in station CSV. Add a demand metric (total_sessions / avg_sessions_per_month / total_energy_kWh).")
print("Using target column:", target_col)

# ---------- 3) Build station-level features ----------
session_pts = sessions[[s_lat, s_lon]].dropna().to_numpy()
station_pts = stations[[e_lat, e_lon]].dropna().to_numpy()

# nearby session counts
stations['nearby_sessions_2km'] = haversine_counts_within(station_pts, session_pts, NEARBY_RADIUS_KM) if session_pts.size and station_pts.size else 0

# competitors & distance to nearest
stations['competitors_within_2km'] = haversine_counts_within(station_pts, station_pts, NEARBY_RADIUS_KM) - 1 if station_pts.size and station_pts.size else 0
if station_pts.shape[0] > 1:
    tree = BallTree(np.radians(station_pts), metric='haversine')
    dist_rad, ind = tree.query(np.radians(station_pts), k=2)
    stations['distance_to_nearest_station_km'] = dist_rad[:,1] * 6371.0
else:
    stations['distance_to_nearest_station_km'] = np.inf

# include other numeric columns automatically (exclude target)
numeric_cols = stations.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [c for c in numeric_cols if c != target_col]
if len(feature_columns) == 0:
    # fallback
    stations['nearby_sessions_2km'] = stations.get('nearby_sessions_2km', 0)
    feature_columns = ['nearby_sessions_2km']

print("Feature columns used:", feature_columns)

# training dataset
train = stations[feature_columns + [target_col]].dropna(subset=[target_col]).copy()
X = train[feature_columns].fillna(0)
y = train[target_col].values
print("Training rows:", len(train))

# ---------- 4) Setup models (with scaling for linear-type) ----------
models = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("lasso", Lasso())]),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, random_state=RANDOM_STATE)
}
if xgb_available:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        tree_method="hist",
        verbosity=0
    )

# ---------- 5) Cross-validate and compare ----------
cv = KFold(n_splits=min(N_FOLDS, max(2, len(X)//2)), shuffle=True, random_state=RANDOM_STATE)
results = []
for name, model in models.items():
    try:
        scores_rmse = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
        scores_r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
        results.append({
            "model": name,
            "rmse_mean": float(scores_rmse.mean()),
            "rmse_std": float(scores_rmse.std()),
            "r2_mean": float(scores_r2.mean()),
            "r2_std": float(scores_r2.std())
        })
        print(f"{name}: RMSE={scores_rmse.mean():.4f} (+/-{scores_rmse.std():.4f}), R2={scores_r2.mean():.4f}")
    except Exception as e:
        print(f"{name} failed during CV: {e}")
        results.append({"model": name, "rmse_mean": np.nan, "rmse_std": np.nan, "r2_mean": np.nan, "r2_std": np.nan})

results_df = pd.DataFrame(results).sort_values("rmse_mean", na_position='last').reset_index(drop=True)
results_df.to_csv(OUTPUT_MODEL_COMP, index=False)
print("\nModel comparison saved to", OUTPUT_MODEL_COMP)

# pick best by lowest RMSE (if tie, top R2)
best_row = results_df.dropna(subset=["rmse_mean"]).iloc[0]
best_name = best_row["model"]
best_r2 = best_row["r2_mean"]
print(f"\nBest model chosen (by RMSE): {best_name} | CV R² = {best_r2:.4f} | Accuracy = {best_r2*100:.2f}%")

# ---------- 6) Fit best model on full train data and save ----------
best_model = models[best_name]
best_model.fit(X, y)
joblib.dump(best_model, OUTPUT_MODEL)
print("Saved best model to", OUTPUT_MODEL)

# ---------- 7) Generate candidate sites from session data ----------
sessions['lat_round'] = sessions[s_lat].round(4)
sessions['lon_round'] = sessions[s_lon].round(4)
candidates = sessions.groupby(['lat_round','lon_round']).size().reset_index(name='session_count')
cand_pts = candidates[['lat_round','lon_round']].to_numpy()
exist_pts = stations[[e_lat, e_lon]].to_numpy()

candidates['min_dist_km'] = haversine_balltree_min_dist_km(cand_pts, exist_pts) if exist_pts.size else np.inf
candidates['nearby_sessions_2km'] = haversine_counts_within(cand_pts, session_pts, NEARBY_RADIUS_KM) if session_pts.size else 0
candidates['competitors_within_2km'] = haversine_counts_within(cand_pts, exist_pts, NEARBY_RADIUS_KM) if exist_pts.size else 0

# build candidate features aligned with feature_columns (proxy mapping from nearest station for extra numeric features)
cand_feat = pd.DataFrame(0, index=candidates.index, columns=feature_columns)
for col in ['nearby_sessions_2km','competitors_within_2km','min_dist_km','distance_to_nearest_station_km']:
    if col in candidates.columns:
        cand_feat[col] = candidates[col].values
    else:
        cand_feat[col] = np.zeros(len(candidates))

try:
    if exist_pts.size:
        tree = BallTree(np.radians(exist_pts), metric='haversine')
        _, idx = tree.query(np.radians(cand_pts), k=1)
        idx = idx.flatten()
        station_numeric = stations[feature_columns].reset_index(drop=True)
        mapped = station_numeric.iloc[idx].reset_index(drop=True)
        for c in feature_columns:
            if c in mapped.columns and c not in ['nearby_sessions_2km','competitors_within_2km','distance_to_nearest_station_km','min_dist_km']:
                cand_feat[c] = mapped[c].values
except Exception:
    pass

cand_feat = cand_feat.fillna(0)[feature_columns]

# ---------- 8) Predict candidate demand and select top near & far ----------
candidates['predicted_demand'] = best_model.predict(cand_feat)

near_bucket = candidates[candidates['min_dist_km'] <= MAX_NEAR_DISTANCE_KM].sort_values('predicted_demand', ascending=False).head(TOP_N_PER_BUCKET).assign(bucket='near')
far_bucket = candidates[candidates['min_dist_km'] >= MIN_DISTANCE_FOR_FAR_KM].sort_values('predicted_demand', ascending=False).head(TOP_N_PER_BUCKET).assign(bucket='far')

combined = pd.concat([near_bucket, far_bucket]).reset_index(drop=True)
combined.to_csv(OUTPUT_SUGGESTIONS, index=False)
print("Saved suggestions to", OUTPUT_SUGGESTIONS)

# ---------- 9) Save map ----------
center = [sessions[s_lat].mean(), sessions[s_lon].mean()]
m = folium.Map(location=center, zoom_start=11)

# existing stations -> green markers
for _, r in stations.iterrows():
    folium.Marker(
        location=[r[e_lat], r[e_lon]],
        icon=folium.Icon(color="green", icon="ok-sign"),
        popup=f"Existing Station: {r.get('station_name','Unknown')} | {target_col}: {r.get(target_col)}"
    ).add_to(m)

# suggested points -> red markers (size scaled by predicted demand)
maxpred = combined['predicted_demand'].max() if not combined.empty else 1.0
for _, r in combined.iterrows():
    folium.Marker(
        location=[r['lat_round'], r['lon_round']],
        icon=folium.Icon(color="red", icon="flag"),
        popup=f"Suggested New Station | Predicted demand: {r['predicted_demand']:.2f} | "
              f"Bucket: {r['bucket']} | Dist from nearest: {r['min_dist_km']:.2f} km"
    ).add_to(m)

m.save(OUTPUT_MAP)
print("Saved map to", OUTPUT_MAP)

# ---------- 10) Summary print ----------
print("\nModel comparison (top rows):")
print(results_df.head())
print(f"\nBest model: {best_name} | CV RMSE: {best_row['rmse_mean']:.4f} | CV R²: {best_row['r2_mean']:.4f} ({best_row['r2_mean']*100:.2f}%)")


Session coords: latitude,longitude | Station coords: latitude,longitude
Rows -> sessions: 314, stations: 236
Using target column: sessions_count
Feature columns used: ['energy_consumed_kwh_sum', 'energy_consumed_kwh_mean', 'charging_duration_hours_mean', 'charging_rate_kw_mean', 'latitude', 'longitude', 'stations_within_2km', 'nearby_sessions_2km', 'competitors_within_2km', 'distance_to_nearest_station_km']
Training rows: 236
LinearRegression: RMSE=0.6045 (+/-0.0589), R2=0.9598
Ridge: RMSE=0.6184 (+/-0.0729), R2=0.9578
Lasso: RMSE=2.9497 (+/-0.2925), R2=0.0565
RandomForest: RMSE=0.2925 (+/-0.0743), R2=0.9893
GradientBoosting: RMSE=0.2380 (+/-0.0489), R2=0.9932
XGBoost: RMSE=0.3565 (+/-0.0709), R2=0.9852

Model comparison saved to regression_model_comparison_results.csv

Best model chosen (by RMSE): GradientBoosting | CV R² = 0.9932 | Accuracy = 99.32%
Saved best model to Model.pkl
Saved suggestions to Predicted_Locations.csv
Saved map to Predicted_Locations_Map.html

Model comparison (